In [3]:
import pandas as pd
df=pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1', engine='python', quotechar='"', on_bad_lines='skip')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
negative    7613
positive    7397
Name: count, dtype: int64


In [5]:
df['sentiment'] = df['sentiment'].map({'positive':1,'negative':0})

In [4]:
negation_words = [
    "Not Good","Not bad","not great","don't like","didn't like","never liked","wasn't good", "no good"
    ]

def clean_text(text):
  text = text.lower()

  text = re.sub(r"[^a-zAA-Z\s']"," ",text)

  for phrase in negation_words:
    text = text.replace(phrase,phrase.replace(" ",""))

  return text

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train , y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size = 0.2,
    random_state = 42)

In [7]:

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token=""
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding='post'
)

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout = 0.3,recurrent_dropout = 0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

In [10]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
151/151 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - accuracy: 0.5186 - loss: 0.6914 - val_accuracy: 0.5462 - val_loss: 0.6816
Epoch 2/5
151/151 ━━━━━━━━━━━━━━━━━━━━ 149s 990ms/step - accuracy: 0.5817 - loss: 0.6439 - val_accuracy: 0.5600 - val_loss: 0.6592
Epoch 3/5
151/151 ━━━━━━━━━━━━━━━━━━━━ 156s 1s/step - accuracy: 0.6182 - loss: 0.5712 - val_accuracy: 0.5812 - val_loss: 0.6688
Epoch 4/5
151/151 ━━━━━━━━━━━━━━━━━━━━ 196s 993ms/step - accuracy: 0.6408 - loss: 0.5266 - val_accuracy: 0.5799 - val_loss: 0.7147
Epoch 5/5
151/151 ━━━━━━━━━━━━━━━━━━━━ 149s 989ms/step - accuracy: 0.6442 - loss: 0.5019 - val_accuracy: 0.5695 - val_loss: 0.7767


In [13]:
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.2f}%")

Test Accuracy: 57.59%


In [12]:
import re

def predict_sentiment(review):
    review = clean_text(review)

    seq = tokenizer.texts_to_sequences([review])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    prediction = model.predict(padded, verbose=0)[0][0]

    print("\nReview:", review)
    print("Score:", prediction)

    if prediction >= 0.5:
        print("Sentiment: Positive")
    else:
        print("Sentiment: Negative")


# Test the function
predict_sentiment("this was amazing")


Review: this was amazing
Score: 0.5141818
Sentiment: Positive
